In [ ]:
!pip install tokenizers

In [1]:
"""
tokenizer.py
------------
BPE tokenizer using HuggingFace tokenizers library.
Trained on WikiText-103, English + printable ASCII only.

Install:
    pip install tokenizers datasets

Usage:
    python tokenizer.py --vocab_size 4096 --out_dir tokenizer/
"""

import re
import os
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder


# ── Special tokens ────────────────────────────────────────────────────────────

SPECIAL_TOKENS = [
    # Core
    "<pad>",          # 0  - padding
    "<unk>",          # 1  - unknown character
    "<bos>",          # 2  - begin sequence
    "<eos>",          # 3  - end sequence
    # Chat roles
    "<|system|>",     # 4  - system message boundary
    "<|user|>",       # 5  - user message boundary
    "<|assistant|>",  # 6  - assistant message boundary
    # Reserved for agents (assign meaning at inference time)
    "<|reserved_0|>", # 7
    "<|reserved_1|>", # 8
    "<|reserved_2|>", # 9
    "<|reserved_3|>", # 10
    "<|reserved_4|>", # 11
    "<|reserved_5|>", # 12
    "<|reserved_6|>", # 13
    "<|reserved_7|>", # 14
    "<|reserved_8|>", # 15
    "<|reserved_9|>", # 16
]

# Core token IDs
PAD_ID       = 0
UNK_ID       = 1
BOS_ID       = 2
EOS_ID       = 3

# Chat role token IDs
SYSTEM_ID    = 4
USER_ID      = 5
ASSISTANT_ID = 6

# Agent reserved token IDs
RESERVED_IDS = list(range(7, 17))   # 7–16 inclusive


# ── Clean text: keep only keyboard-typeable characters ────────────────────────

def clean(text: str) -> str:
    """Keep only printable ASCII (32-126). Normalize whitespace."""
    filtered = "".join(c for c in text if 32 <= ord(c) <= 126)
    return re.sub(r"\s+", " ", filtered).strip()


# ── Load + clean all datasets ─────────────────────────────────────────────────
 
DATASET_CAPS = {
    "Salesforce/wikitext":        200_000,   # ~WikiText-103 full train, small anyway
    "bookcorpus/bookcorpus":      300_000,   # books, cap at 300k lines
    "Skylion007/openwebtext":     200_000,   # web text, just take first 200k lines
    "roneneldan/TinyStories":     200_000,   # simple stories
}
 
def load_all_datasets(smoke_test: bool = False) -> list:
    from datasets import load_dataset
 
    sources = [
        ("Salesforce/wikitext",     "wikitext-103-v1",  "train", "text"),
        ("bookcorpus/bookcorpus",   None,               "train", "text"),
        ("Skylion007/openwebtext",  None,               "train", "text"),
        ("roneneldan/TinyStories",  None,               "train", "text"),
    ]
 
    all_texts = []
 
    for name, config, split, field in sources:
        try:
            print(f"Loading {name}...")
            cap = 2000 if smoke_test else DATASET_CAPS[name]
 
            # streaming=True avoids downloading the whole dataset —
            # we just take the first `cap` rows and stop
            ds = load_dataset(name, config, split=split,
                              streaming=True, trust_remote_code=True)
 
            texts = []
            for row in ds:
                line = clean(row[field])
                if len(line) >= 10:
                    texts.append(line)
                if len(texts) >= cap:
                    break
 
            print(f"  {name}: {len(texts):,} lines")
            all_texts.extend(texts)
 
        except Exception as e:
            print(f"  WARNING: Could not load {name} — {e}. Skipping.")
 
    print(f"\nTotal lines: {len(all_texts):,}")
    return all_texts
 

 
 
 
# ── Train BPE tokenizer ───────────────────────────────────────────────────────
 
def train(texts: list, vocab_size: int = 4096, out_dir: str = "tokenizer/"):
    os.makedirs(out_dir, exist_ok=True)
    corpus_path = os.path.join(out_dir, "corpus.txt")
 
    print(f"Writing corpus to {corpus_path}...")
    with open(corpus_path, "w", encoding="utf-8") as f:
        f.write("\n".join(texts))
 
    tokenizer = Tokenizer(BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
    tokenizer.decoder = ByteLevelDecoder()
 
    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=SPECIAL_TOKENS,
        show_progress=True,
    )
 
    print(f"Training BPE (vocab_size={vocab_size})...")
    tokenizer.train([corpus_path], trainer)
 
    save_path = os.path.join(out_dir, "tokenizer.json")
    tokenizer.save(save_path)
    # delete corpus after training, it's no longer needed
    os.remove(corpus_path)
    print("Corpus file deleted.")
    print(f"Saved -> {save_path}  (vocab: {tokenizer.get_vocab_size()} tokens)")
 
    return tokenizer


texts     = load_all_datasets(smoke_test=False)
tokenizer = train(texts, vocab_size=4096, out_dir="tokenizer/")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Salesforce/wikitext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading Salesforce/wikitext...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'bookcorpus/bookcorpus' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  Salesforce/wikitext: 200,000 lines
Loading bookcorpus/bookcorpus...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Skylion007/openwebtext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading Skylion007/openwebtext...


Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'roneneldan/TinyStories' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  Skylion007/openwebtext: 200,000 lines
Loading roneneldan/TinyStories...
  roneneldan/TinyStories: 200,000 lines

Total lines: 600,000
Writing corpus to tokenizer/corpus.txt...
Training BPE (vocab_size=4096)...
Corpus file deleted.
Saved -> tokenizer/tokenizer.json  (vocab: 4096 tokens)


In [6]:
# ── Quick test ────────────────────────────────────────────────────────────────

def test(tokenizer: Tokenizer):
    print("\n--- Round-trip test ---")
    tests = [
        " The quick brown fox jumps over the lazy dog.",
        "Hello world! 42 + 3.14 = something",
        "Don't forget semi-colons; they matter!",
        "x = (a + b) * c / 2",
        "Medhavi monish Kunal"
    ]
    for text in tests:
        enc     = tokenizer.encode(text)
        decoded = tokenizer.decode(enc.ids)
        print(f"  Input  : {text}")
        print(f"  Tokens : {enc.tokens}")
        print(f"  Decoded: {decoded}\n")

    print("--- Chat formatting test ---")
    messages = [
        {"role": "system",    "content": "You are a helpful assistant."},
        {"role": "user",      "content": "What is 2 + 2?"},
        {"role": "assistant", "content": "4"},
    ]
    ids    = format_chat(tokenizer, messages)
    vocab  = {v: k for k, v in tokenizer.get_vocab().items()}
    tokens = [vocab.get(i, "?") for i in ids]
    print(f"  Messages : {messages}")
    print(f"  Token IDs: {ids}")
    print(f"  Tokens   : {tokens}\n")

    print("--- Special token IDs ---")
    vocab = tokenizer.get_vocab()
    for tok in SPECIAL_TOKENS:
        print(f"  {tok:20s} → {vocab[tok]}")


# ── Chat formatting helper ────────────────────────────────────────────────────

def format_chat(tokenizer: Tokenizer, messages: list) -> list:
    """
    Format a list of chat messages into a flat token ID sequence.

    Parameters
    ----------
    tokenizer : trained Tokenizer
    messages  : list of dicts with keys 'role' and 'content'
                role must be 'system', 'user', or 'assistant'

    Returns
    -------
    list of int token IDs

    Example
    -------
    messages = [
        {"role": "system",    "content": "You are a helpful assistant."},
        {"role": "user",      "content": "What is 2 + 2?"},
        {"role": "assistant", "content": "4"},
    ]
    ids = format_chat(tokenizer, messages)
    # <bos> <|system|> You are... <|user|> What is... <|assistant|> 4 <eos>
    """
    role_token = {
        "system":    "<|system|>",
        "user":      "<|user|>",
        "assistant": "<|assistant|>",
    }

    vocab = tokenizer.get_vocab()
    ids   = [BOS_ID]

    for msg in messages:
        role    = msg["role"]
        content = msg["content"]

        if role not in role_token:
            raise ValueError(f"Unknown role: {role}. Must be system/user/assistant.")

        # Add role boundary token
        ids.append(vocab[role_token[role]])

        # Encode content (no bos/eos wrapping — we handle that manually)
        enc = tokenizer.encode(content)
        ids.extend(enc.ids)

    ids.append(EOS_ID)
    return ids


# ── CLI ───────────────────────────────────────────────────────────────────────
test(tokenizer)


--- Round-trip test ---
  Input  :  The quick brown fox jumps over the lazy dog.
  Tokens : ['ĠThe', 'Ġquick', 'Ġbro', 'wn', 'Ġf', 'ox', 'Ġjump', 's', 'Ġover', 'Ġthe', 'Ġl', 'az', 'y', 'Ġdog', '.']
  Decoded:  The quick brown fox jumps over the lazy dog.

  Input  : Hello world! 42 + 3.14 = something
  Tokens : ['H', 'ell', 'o', 'Ġworld', '!', 'Ġ4', '2', 'Ġ+', 'Ġ3', '.', '14', 'Ġ=', 'Ġsomething']
  Decoded: Hello world! 42 + 3.14 = something

  Input  : Don't forget semi-colons; they matter!
  Tokens : ['Don', "'t", 'Ġfor', 'get', 'Ġs', 'em', 'i', '-', 'c', 'ol', 'ons', ';', 'Ġthey', 'Ġmatter', '!']
  Decoded: Don't forget semi-colons; they matter!

  Input  : x = (a + b) * c / 2
  Tokens : ['x', 'Ġ=', 'Ġ(', 'a', 'Ġ+', 'Ġb', ')', 'Ġ*', 'Ġc', 'Ġ/', 'Ġ2']
  Decoded: x = (a + b) * c / 2

  Input  : Medhavi monish Kunal
  Tokens : ['M', 'ed', 'h', 'av', 'i', 'Ġmon', 'ish', 'ĠK', 'un', 'al']
  Decoded: Medhavi monish Kunal

--- Chat formatting test ---
  Messages : [{'role': 'system', 'con